# Concatenation Transformation Test

## What is tested

`reshape_superoperators_list_to_direct_sum_basis` assembles the superoperators of independent
subsystems into one superoperator acting on the direct-sum Liouville space. This notebook
checks that the combined superoperator acts on a block-diagonal density matrix exactly as
the individual superoperators act on their own blocks. All density matrices use row-major
vectorization.

## List of tests

1. `test_two_equal_subsystems`: two 2x2 subsystems.
2. `test_single_subsystem`: one 3x3 subsystem, where the combined superoperator must act like the original one.
3. `test_three_different_subsystems`: subsystems of sizes 2x2, 3x3 and 1x1.

## How to run

Run all cells. The last cell calls `run_tests()`. Each test prints one line when it
passes and raises `AssertionError` when it fails.

In [1]:
import os
import sys

import torch

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

from mars.population.transform import (
    Liouvilleator,
    reshape_superoperators_list_to_direct_sum_basis,
)

## Helper functions

In [2]:
def check_action_on_block_diagonal_state(dims: list[int]) -> None:
    """Compare the combined superoperator with block-by-block action on a random block-diagonal state."""
    superops = [torch.randn(dim * dim, dim * dim) for dim in dims]
    total_superop = reshape_superoperators_list_to_direct_sum_basis(superops)

    total_dim = sum(dims)
    assert total_superop.shape == (total_dim * total_dim, total_dim * total_dim), (
        f"Wrong shape of the combined superoperator: {total_superop.shape}"
    )

    blocks = [torch.randn(dim, dim) for dim in dims]
    result = total_superop @ Liouvilleator.vec(torch.block_diag(*blocks))

    expected_blocks = [
        Liouvilleator.unvec(superop @ Liouvilleator.vec(block))
        for superop, block in zip(superops, blocks)
    ]
    expected = Liouvilleator.vec(torch.block_diag(*expected_blocks))

    assert torch.allclose(result, expected, atol=1e-6), (
        f"Combined superoperator acts incorrectly for subsystem sizes {dims}"
    )

## Tests

In [3]:
def test_two_equal_subsystems():
    """Check the direct sum of two subsystems of equal size (2x2 and 2x2)."""
    torch.manual_seed(42)
    check_action_on_block_diagonal_state([2, 2])
    print("Two equal subsystems test passed (blocks 2x2 and 2x2 evolve independently)")


def test_single_subsystem():
    """Check that the direct sum of one subsystem acts like the original superoperator."""
    torch.manual_seed(42)
    dim = 3
    superop = torch.randn(dim * dim, dim * dim)
    total_superop = reshape_superoperators_list_to_direct_sum_basis([superop])
    assert total_superop.shape == (dim * dim, dim * dim), (
        f"Wrong shape of the combined superoperator: {total_superop.shape}"
    )

    vec_rho = Liouvilleator.vec(torch.randn(dim, dim))
    assert torch.allclose(total_superop @ vec_rho, superop @ vec_rho), (
        "A single subsystem must keep its superoperator unchanged"
    )
    print("Single subsystem test passed (the combined superoperator equals the original)")


def test_three_different_subsystems():
    """Check the direct sum of three subsystems of different sizes (2x2, 3x3 and 1x1)."""
    torch.manual_seed(42)
    check_action_on_block_diagonal_state([2, 3, 1])
    print("Three different subsystems test passed (blocks 2x2, 3x3 and 1x1 evolve independently)")

## Run all tests

In [4]:
def run_tests():
    """Run every test of this notebook."""
    tests = [test_two_equal_subsystems, test_single_subsystem, test_three_different_subsystems]
    for test in tests:
        test()
    print(f"\nAll {len(tests)} concatenation transformation tests passed.")


run_tests()

Two equal subsystems test passed (blocks 2x2 and 2x2 evolve independently)
Single subsystem test passed (the combined superoperator equals the original)
Three different subsystems test passed (blocks 2x2, 3x3 and 1x1 evolve independently)

All 3 concatenation transformation tests passed.
